In [ ]:
# libraries and packages


In [2]:
import pandas as pd

#  Step 1. Load dataset (full file)
heart = pd.read_csv("heart_rate.csv")
print(f" Loaded dataset with shape: {heart.shape}")

# Step 2. Define  key columns for deduplication
key_cols = ["id", "study_interval", "is_weekend", "day_in_study", "timestamp"]

#  Step 3. Remove duplicates (keep the first occurrence only) 
heart_clean = heart.drop_duplicates(subset=key_cols, keep="first")

#  Step 4. Summary of results
total_rows = len(heart)
cleaned_rows = len(heart_clean)
removed_rows = total_rows - cleaned_rows

print(f" Cleaned dataset shape: {heart_clean.shape}")
print(f" Rows removed: {removed_rows}")
print(f" Remaining duplicates: {heart_clean.duplicated(subset=key_cols).sum()}")

# Step 5. Save cleaned dataset
heart_clean.to_csv("heart_rate_cleaned.csv", index=False)
print("Cleaned file saved as: heart_rate_cleaned.csv")


 Loaded dataset with shape: (63100276, 7)
 Cleaned dataset shape: (62043919, 7)
 Rows removed: 1056357
 Remaining duplicates: 0
Cleaned file saved as: heart_rate_cleaned.csv


In [2]:

import pandas as pd
heart_rate_cleaned = pd.read_csv("heart_rate_cleaned.csv")
print(heart_rate_cleaned.head(2))


   id  study_interval  is_weekend  day_in_study timestamp  bpm  confidence
0   1            2022        True             1  01:22:09   75           1
1   1            2022        True             1  01:22:14   79           1


In [3]:
import pandas as pd
import numpy as np

# --- Step 1. Load and Prepare Data ---
heart_rate_cleaned = pd.read_csv("heart_rate_cleaned.csv")
heart_rate_cleaned["timestamp"] = pd.to_datetime(heart_rate_cleaned["timestamp"], errors="coerce")

# --- Step 2. Drop the unnecessary column ---
heart_rate_cleaned = heart_rate_cleaned.drop(columns=["confidence"])

# --- Step 3. Compute time in minutes from midnight ---
heart_rate_cleaned["time_in_minutes"] = (
    heart_rate_cleaned["timestamp"].dt.hour * 60 + heart_rate_cleaned["timestamp"].dt.minute
)

# --- Step 4. Define 2-hour checkpoints ---
checkpoints = pd.date_range("00:00", "22:00", freq="2H").time
checkpoint_minutes = [t.hour * 60 + t.minute for t in checkpoints]


# --- Step 5. Define directional aggregation function ---
def get_closest_bpm(group):
    results = {}
    group_times = group["time_in_minutes"]

    for i in range(len(checkpoints)):
        current_time = checkpoints[i]
        current_min = checkpoint_minutes[i]
        prefix = f"{current_time.strftime('%-I%p').lower()}"
        next_min = checkpoint_minutes[(i + 1) % len(checkpoints)]

        # Directional window [current_min, next_min)
        if current_min < next_min:
            filtered_indices = group_times[
                (group_times >= current_min) & (group_times < next_min)
            ].index
        else:
            filtered_indices = group_times[group_times >= current_min].index

        if not filtered_indices.empty:
            filtered_group = group.loc[filtered_indices]
            diffs = filtered_group["time_in_minutes"] - current_min
            idx_min_diff = diffs.idxmin()
            closest = filtered_group.loc[idx_min_diff]

            results[f"{prefix}_bpm"] = closest["bpm"]
        else:
            results[f"{prefix}_bpm"] = np.nan

    return pd.Series(results)


# --- Step 6. Group by participant/day (exclude is_weekend) ---
agg_heart = (
    heart_rate_cleaned.groupby(["id", "study_interval", "day_in_study"], group_keys=False)
               .apply(get_closest_bpm)
               .reset_index()
)

# --- Step 7. Merge back correct is_weekend flag ---
agg_heart = agg_heart.merge(
    heart_rate_cleaned[["id", "study_interval", "day_in_study", "is_weekend"]].drop_duplicates(),
    on=["id", "study_interval", "day_in_study"],
    how="left"
)

# --- Step 8. Reorder columns (place is_weekend after day_in_study) ---
cols = agg_heart.columns.tolist()
cols.insert(cols.index("day_in_study") + 1, cols.pop(cols.index("is_weekend")))
agg_heart = agg_heart[cols]

# --- Step 9. Sort chronologically ---
agg_heart = agg_heart.sort_values(["id", "study_interval", "day_in_study"])

# --- Step 10. Save Final Transformed File ---
output_path = "heart_rate_2hr.csv"
agg_heart.to_csv(output_path, index=False)

print(f"\nFinal heart rate file saved to: {output_path}")
print("\nPreview of transformed data:")
print(agg_heart.head(2))


/var/tmp/ipykernel_4710/1309664461.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  heart_rate_cleaned["timestamp"] = pd.to_datetime(heart_rate_cleaned["timestamp"], errors="coerce")
/var/tmp/ipykernel_4710/1309664461.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  checkpoints = pd.date_range("00:00", "22:00", freq="2H").time
/var/tmp/ipykernel_4710/1309664461.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_closest_bpm)



Final heart rate file saved to: heart_rate_2hr.csv

Preview of transformed data:
   id  study_interval  day_in_study  is_weekend  12am_bpm  2am_bpm  4am_bpm  \
0   1            2022             1        True      75.0      NaN     84.0   
1   1            2022             2       False      87.0     73.0     95.0   

   6am_bpm  8am_bpm  10am_bpm  12pm_bpm  2pm_bpm  4pm_bpm  6pm_bpm  8pm_bpm  \
0     73.0     68.0      67.0      75.0     71.0     83.0     57.0    101.0   
1     76.0     75.0     167.0     119.0     90.0     98.0     93.0      NaN   

   10pm_bpm  
0      85.0  
1       NaN  


In [ ]:
# checking the results: In original csv, the data does check out for id :1, day :1, and time 12 am  cntd.

In [1]:
heart_rate_2hr.info()
heart_rate_2hr.shape()

NameError: name 'heart_rate_2hr' is not defined

In [2]:
import pandas as pd

heart_rate_2hr = pd.read_csv("heart_rate_2hr.csv")
heart_rate_2hr.info()
print(heart_rate_2hr.shape)
# heart_rate_csv.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5443 entries, 0 to 5442
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              5443 non-null   int64  
 1   study_interval  5443 non-null   int64  
 2   day_in_study    5443 non-null   int64  
 3   is_weekend      5443 non-null   bool   
 4   12am_bpm        5119 non-null   float64
 5   2am_bpm         5212 non-null   float64
 6   4am_bpm         5254 non-null   float64
 7   6am_bpm         5283 non-null   float64
 8   8am_bpm         5257 non-null   float64
 9   10am_bpm        5250 non-null   float64
 10  12pm_bpm        5268 non-null   float64
 11  2pm_bpm         5226 non-null   float64
 12  4pm_bpm         5177 non-null   float64
 13  6pm_bpm         5160 non-null   float64
 14  8pm_bpm         5152 non-null   float64
 15  10pm_bpm        5149 non-null   float64
dtypes: bool(1), float64(12), int64(3)
memory usage: 643.3 KB
(5443, 16)


In [5]:
missing_count = heart_rate_2hr.isnull().sum()
print(missing_count)

id                  0
study_interval      0
day_in_study        0
is_weekend          0
12am_bpm          324
2am_bpm           231
4am_bpm           189
6am_bpm           160
8am_bpm           186
10am_bpm          193
12pm_bpm          175
2pm_bpm           217
4pm_bpm           266
6pm_bpm           283
8pm_bpm           291
10pm_bpm          294
dtype: int64


In [7]:
heart_rate_2hr.columns.tolist()

['id',
 'study_interval',
 'day_in_study',
 'is_weekend',
 '12am_bpm',
 '2am_bpm',
 '4am_bpm',
 '6am_bpm',
 '8am_bpm',
 '10am_bpm',
 '12pm_bpm',
 '2pm_bpm',
 '4pm_bpm',
 '6pm_bpm',
 '8pm_bpm',
 '10pm_bpm']

In [ ]:
# for Heart rate variabolity I had started the transformation on Colab actually, like duplicate and all, so I am going to upload the clean data for Heart Variability here in the team2project, for reference the codes I used are below. 

In [ ]:
# hrv = pd.read_csv("/content/drive/MyDrive/BA878/Dataset/physionet.org/files/mcphases/1.0.0/heart_rate_variability_details.csv")

# #  Define the key columns 
# key_cols = ["id", "study_interval", "day_in_study", "timestamp"]

# #  Remove duplicates (keep only first occurrence) ---
# hrv_clean = hrv.drop_duplicates(subset=key_cols, keep="first")

# # Summary statistics
# total_rows = len(hrv)
# duplicates_removed = total_rows - len(hrv_clean)
# unique_rows_kept = len(hrv_clean)

# print(f" Original rows: {total_rows}")
# print(f" Cleaned rows (first occurrences kept): {unique_rows_kept}")
# print(f" Duplicates removed: {duplicates_removed}")
# print(f"Remaining duplicates after cleaning: {hrv_clean.duplicated(subset=key_cols).sum()}")

In [ ]:
# Original rows: 436262
# Cleaned rows (first occurrences kept): 427918
#  Duplicates removed: 8344
# Remaining duplicates after cleaning: 0

In [ ]:
# Transforming the Heart Variability dataset

# import pandas as pd
# import numpy as np

# #  Step 1. Load and Prepare Data 
# hrv_clean = pd.read_csv("/content/drive/MyDrive/BA878/Dataset/physionet.org/files/mcphases/1.0.0/heartratevariability_details_cleaned.csv")
# hrv_clean["timestamp"] = pd.to_datetime(hrv_clean["timestamp"], errors="coerce")

# # Calculate time in minutes from midnight
# hrv_clean["time_in_minutes"] = (
#     hrv_clean["timestamp"].dt.hour * 60 + hrv_clean["timestamp"].dt.minute
# )

# # Step 2. Define 2-hour Checkpoints 
# checkpoints = pd.date_range("00:00", "22:00", freq="2H").time
# checkpoint_minutes = [t.hour * 60 + t.minute for t in checkpoints]


# # Step 3. Define Directional Aggregation Function 
# def get_closest_records_directional(group):
#     results = {}
#     group_times = group["time_in_minutes"]

#     for i in range(len(checkpoints)):
#         current_time = checkpoints[i]
#         current_min = checkpoint_minutes[i]
#         prefix = f"{current_time.strftime('%-I%p').lower()}"
#         next_min = checkpoint_minutes[(i + 1) % len(checkpoints)]

#         # Define non-overlapping window [current_min, next_min)
#         if current_min < next_min:
#             filtered_indices = group_times[
#                 (group_times >= current_min) & (group_times < next_min)
#             ].index
#         else:
#             # Wrap-around for last interval (10pm to midnight)
#             filtered_indices = group_times[group_times >= current_min].index

#         if not filtered_indices.empty:
#             filtered_group = group.loc[filtered_indices]
#             diffs = filtered_group["time_in_minutes"] - current_min
#             idx_min_diff = diffs.idxmin()
#             closest = filtered_group.loc[idx_min_diff]

#             results[f"{prefix}_rmssd"] = closest["rmssd"]
#             results[f"{prefix}_lf"] = closest["low_frequency"]
#             results[f"{prefix}_hf"] = closest["high_frequency"]
#         else:
#             results[f"{prefix}_rmssd"] = np.nan
#             results[f"{prefix}_lf"] = np.nan
#             results[f"{prefix}_hf"] = np.nan

#     return pd.Series(results)


# #  Step 4. Group by ID, study_interval, day_in_study (exclude is_weekend) 
# agg_hrv_directional = (
#     hrv_clean.groupby(["id", "study_interval", "day_in_study"], group_keys=False)
#              .apply(get_closest_records_directional)
#              .reset_index()
# )

# #  Step 5. Merge back the correct is_weekend flag for each day
# agg_hrv_directional = agg_hrv_directional.merge(
#     hrv_clean[["id", "study_interval", "day_in_study", "is_weekend"]].drop_duplicates(),
#     on=["id", "study_interval", "day_in_study"],
#     how="left"
# )

# # Step 6. Reorder columns (move is_weekend after day_in_study) 
# cols = agg_hrv_directional.columns.tolist()
# cols.insert(cols.index("day_in_study") + 1, cols.pop(cols.index("is_weekend")))
# agg_hrv_directional = agg_hrv_directional[cols]

# # Step 7. Sort by ID, study_interval, and day_in_study 
# agg_hrv_directional_new1 = agg_hrv_directional.sort_values(["id", "study_interval", "day_in_study"])

# #  Step 8. Optional Validation 
# # Check if any day has more than one weekend flag (should be 1 unique value per day)
# check = (
#     hrv_clean.groupby(["id", "study_interval", "day_in_study"])["is_weekend"]
#     .nunique()
#     .reset_index(name="unique_flags")
# )
# multiple_flags = check[check["unique_flags"] > 1]
# if not multiple_flags.empty:
#     print(" Warning: Some days have inconsistent weekend flags:")
#     print(multiple_flags)
# else:
#     print("Each day has a single consistent weekend flag.")

# #  Step 9. Save Final Output 
# output_path = "agg_hrv_directional_new1 .csv"
# agg_hrv_directional_new1.to_csv(output_path, index=False)

# print(f"\nFinal dataset saved to: {output_path}")
# print("\nPreview of transformed data:")
# print(agg_hrv_directional_new1.head(2))


In [ ]:
# Preview of transformed Heart variability data:
#    id  study_interval  day_in_study  is_weekend  12am_rmssd  12am_lf  12am_hf  \
# 0   2            2022             7        True         NaN      NaN      NaN   
# 1   2            2022             8        True      39.195  107.099  362.024   

#    2am_rmssd   2am_lf    2am_hf  ...  4pm_hf  6pm_rmssd  6pm_lf  6pm_hf  \
# 0        NaN      NaN       NaN  ...     NaN        NaN     NaN     NaN   
# 1     63.081  313.021  1001.656  ...     NaN        NaN     NaN     NaN   

#    8pm_rmssd  8pm_lf  8pm_hf  10pm_rmssd  10pm_lf  10pm_hf  
# 0        NaN     NaN     NaN      37.058  817.292  257.772  
# 1        NaN     NaN     NaN      34.976  139.091  242.083  

In [2]:
import pandas as pd

agg_hrv_directional_new1 = pd.read_csv ("agg_hrv_directional_new1 .csv")

hrv_2hr = pd.read_csv("agg_hrv_directional_new1 .csv")

hrv_2hr = agg_hrv_directional_new1.copy()

In [12]:
print("Shape:", hrv_2hr.shape)
hrv_2hr.info()
display(hrv_2hr.head(2))

print(hrv_2hr.columns.tolist())

Shape: (4839, 40)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4839 entries, 0 to 4838
Data columns (total 40 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              4839 non-null   int64  
 1   study_interval  4839 non-null   int64  
 2   day_in_study    4839 non-null   int64  
 3   is_weekend      4839 non-null   bool   
 4   12am_rmssd      3244 non-null   float64
 5   12am_lf         3244 non-null   float64
 6   12am_hf         3244 non-null   float64
 7   2am_rmssd       4365 non-null   float64
 8   2am_lf          4365 non-null   float64
 9   2am_hf          4365 non-null   float64
 10  4am_rmssd       4646 non-null   float64
 11  4am_lf          4646 non-null   float64
 12  4am_hf          4646 non-null   float64
 13  6am_rmssd       4397 non-null   float64
 14  6am_lf          4397 non-null   float64
 15  6am_hf          4397 non-null   float64
 16  8am_rmssd       3095 non-null   float64
 17  8am_lf         

,id,study_interval,day_in_study,is_weekend,12am_rmssd,12am_lf,12am_hf,2am_rmssd,2am_lf,2am_hf,...,4pm_hf,6pm_rmssd,6pm_lf,6pm_hf,8pm_rmssd,8pm_lf,8pm_hf,10pm_rmssd,10pm_lf,10pm_hf
0,2,2022,7,True,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.058,817.292,257.772
1,2,2022,8,True,39.195,107.099,362.024,63.081,313.021,1001.656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.976,139.091,242.083


['id', 'study_interval', 'day_in_study', 'is_weekend', '12am_rmssd', '12am_lf', '12am_hf', '2am_rmssd', '2am_lf', '2am_hf', '4am_rmssd', '4am_lf', '4am_hf', '6am_rmssd', '6am_lf', '6am_hf', '8am_rmssd', '8am_lf', '8am_hf', '10am_rmssd', '10am_lf', '10am_hf', '12pm_rmssd', '12pm_lf', '12pm_hf', '2pm_rmssd', '2pm_lf', '2pm_hf', '4pm_rmssd', '4pm_lf', '4pm_hf', '6pm_rmssd', '6pm_lf', '6pm_hf', '8pm_rmssd', '8pm_lf', '8pm_hf', '10pm_rmssd', '10pm_lf', '10pm_hf']


In [13]:
# reading time_in_heart_rate_zones file 

tihz = pd.read_csv("time_in_heart_rate_zones.csv")

In [15]:
tihz.info()
tihz.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5553 entries, 0 to 5552
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5553 non-null   int64  
 1   study_interval        5553 non-null   int64  
 2   is_weekend            5553 non-null   bool   
 3   day_in_study          5553 non-null   int64  
 4   in_default_zone_3     5553 non-null   float64
 5   in_default_zone_2     5553 non-null   float64
 6   in_default_zone_1     5553 non-null   float64
 7   below_default_zone_1  5553 non-null   float64
dtypes: bool(1), float64(4), int64(3)
memory usage: 309.2 KB


(5553, 8)

In [16]:
# Step 2. Identify duplicates by logical keys
key_cols = ["id", "study_interval", "day_in_study", "is_weekend"]
dup_mask = tihz.duplicated(subset=key_cols, keep=False)
print(f"Found {dup_mask.sum():,} duplicate rows based on {key_cols}.")


Found 172 duplicate rows based on ['id', 'study_interval', 'day_in_study', 'is_weekend'].


In [17]:
# Step 3. Export duplicate rows for inspection
tihz_dupes = tihz.loc[dup_mask].sort_values(key_cols)
tihz_dupes.to_csv("time_in_heart_rate_zones_duplicates.csv", index=False)
print("Exported duplicates to: time_in_heart_rate_zones_duplicates.csv")

Exported duplicates to: time_in_heart_rate_zones_duplicates.csv


In [18]:
# Step 4. Remove duplicates (keep first occurrence)
tihz_clean = tihz.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)
print(f"Cleaned dataset now has {len(tihz_clean):,} rows (removed {len(tihz) - len(tihz_clean):,} duplicates).")

# Step 5. Save cleaned dataset
tihz_clean.to_csv("time_in_heart_rate_zones_cleaned.csv", index=False)
print("Cleaned file saved as: time_in_heart_rate_zones_cleaned.csv")

Cleaned dataset now has 5,450 rows (removed 103 duplicates).
Cleaned file saved as: time_in_heart_rate_zones_cleaned.csv


In [6]:
import pandas as pd
tihz_clean = pd.read_csv("time_in_heart_rate_zones_cleaned.csv")
tihz_clean.columns.tolist()


['id',
 'study_interval',
 'is_weekend',
 'day_in_study',
 'in_default_zone_3',
 'in_default_zone_2',
 'in_default_zone_1',
 'below_default_zone_1']

In [ ]:
# merging files- heart_rate_2hr, tihz_clean, hrv_2hr

In [ ]:
# read the files to define the files - so that we don;t get this is undefined

In [2]:
import pandas as pd 
tihz_clean = pd.read_csv("time_in_heart_rate_zones_cleaned.csv")

In [3]:
heart_rate_2hr = pd.read_csv("heart_rate_2hr.csv")


In [4]:
hrv_2hr = pd.read_csv("agg_hrv_directional_new1 .csv")

In [5]:
print("Shapes before merge:")
print("tihz_clean:", tihz_clean.shape)
print("heart_rate_2hr:", heart_rate_2hr.shape)
print("hrv_2hr:", hrv_2hr.shape)

Shapes before merge:
tihz_clean: (5450, 8)
heart_rate_2hr: (5443, 16)
hrv_2hr: (4839, 40)


In [7]:
# Step 1. Define merge keys (exclude is_weekend for now) 
merge_keys = ["id", "study_interval", "day_in_study"]

#  Step 2. Merge 1: Time in Heart rate zones (time based data) as (base) + Heart Rate (left join) 
merged1 = tihz_clean.merge(
    heart_rate_2hr,
    on=merge_keys,
    how="left"
)
print("\nAfter merging zones + heart_rate_2hr:", merged1.shape)


After merging zones + heart_rate_2hr: (5450, 21)


In [8]:
#  Step 3. Merge 2: Add HRV data to the merged dataset 1 (left join) 
merged_final = merged1.merge(
    hrv_2hr,
    on=merge_keys,
    how="left"
)
print("After adding hrv_2hr:", merged_final.shape)

After adding hrv_2hr: (5450, 58)


In [9]:
#  Step 4. Quick inspection
print("\nPreview of merged dataset:")
print(merged_final.head(5))

print("\nNull value summary (first 10 columns):")
print(merged_final.isnull().sum().head(10))

# Step 5. Save final merged dataset 
output_path = "merged_heart.csv"
merged_final.to_csv(output_path, index=False)
print(f"\n Final merged dataset saved as: {output_path}")


Preview of merged dataset:
   id  study_interval  is_weekend_x  day_in_study  in_default_zone_3  \
0   1            2022          True             1                0.0   
1   1            2022         False             2                5.0   
2   1            2022         False             3                5.0   
3   1            2022         False             4                0.0   
4   1            2022         False             5                8.0   

   in_default_zone_2  in_default_zone_1  below_default_zone_1 is_weekend_y  \
0                0.0              126.0                1036.0         True   
1               82.0              416.0                 512.0        False   
2              119.0              599.0                 368.0        False   
3                0.0              212.0                 613.0        False   
4              123.0              250.0                 308.0        False   

   12am_bpm  ...  4pm_hf  6pm_rmssd  6pm_lf  6pm_hf  8pm_rmssd  8pm_lf

In [11]:
import pandas as pd 
merged_heart = pd.read_csv("merged_heart.csv")


print("Shape of dataset:", merged_heart.shape)
print("\nColumn overview:")
print(merged_heart.info())

print("\nPreview of first 10 rows:")
print(merged_heart.head(10))

Shape of dataset: (5450, 58)

Column overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5450 entries, 0 to 5449
Data columns (total 58 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5450 non-null   int64  
 1   study_interval        5450 non-null   int64  
 2   is_weekend_x          5450 non-null   bool   
 3   day_in_study          5450 non-null   int64  
 4   in_default_zone_3     5450 non-null   float64
 5   in_default_zone_2     5450 non-null   float64
 6   in_default_zone_1     5450 non-null   float64
 7   below_default_zone_1  5450 non-null   float64
 8   is_weekend_y          5439 non-null   object 
 9   12am_bpm              5115 non-null   float64
 10  2am_bpm               5209 non-null   float64
 11  4am_bpm               5254 non-null   float64
 12  6am_bpm               5283 non-null   float64
 13  8am_bpm               5257 non-null   float64
 14  10am_bpm              525